In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('data/drom_archive_cleaned_2018-2025full.csv')

In [ ]:
df.info()

In [ ]:
categorical = ['Тип двигателя', 'Коробка передач', 'Модель' 'Привод', 'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Возраст авто', 'Поколение']

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('num', 'passthrough', numerical)], remainder='drop')
lr_model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])

In [ ]:
y = df['Цена']
X = df.drop('Цена', axis=1)

df['price_stratum'] = pd.qcut(df['Цена'], q=10, labels=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df['price_stratum'])

In [ ]:
lr_model.fit(X_train, y_train)
joblib.dump(lr_model, "models/lr_model.pkl")

In [ ]:
def pred_metrics(model, X_test, y_test, model_name, file_name):
    y_pred = model.predict(X_test) # предсказания на тестовой выборке
    metrics = { # словарь с метриками
        "model_name": model_name,
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred),
        "MAPE": mean_absolute_percentage_error(y_test, y_pred)
    }
    # сохранение в pickle-файл
    with open(f"metrics/{file_name}_metrics.pkl", "wb") as f:
        pickle.dump(metrics, f)

    return metrics

In [ ]:
print(pred_metrics(lr_model, X_test, y_test, 'Linear Regression', 'lr'))

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100, # количесвто деревьев
        random_state=42,
        n_jobs=-1, # задействует все ядра процессора
        max_depth=20, # максимальное количество уровней
        verbose=2 # отображает процесс обучения
    ))
])

In [ ]:
rf_model.fit(X_train, y_train)
joblib.dump(rf_model, "models/rf_model.pkl")

In [ ]:
print(pred_metrics(rf_model, X_test, y_test, 'Random Forest', 'rf'))

In [ ]:
categorical_target = ['Город', 'Модель', 'Метка']      # Target Encoding
categorical_onehot = ['Тип двигателя', 'Коробка передач', 'Привод', 'Тип кузова']  # OneHot

preprocessor = ColumnTransformer(
    transformers=[
        ('target', TargetEncoder(), categorical_target),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_onehot),
        ('num', 'passthrough', numerical)
    ]
)

xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=2000, # количество деревьев
        n_jobs=-1, # задействует все ядра
        random_state=42,
        eval_metric='rmse' # метрика для обучения
    ))
])

In [ ]:
xgb_model.fit(X_train, y_train, regressor__verbose=True)
joblib.dump(xgb_model, "models/xgb_model.pkl")

In [ ]:
print(pred_metrics(xgb_model, X_test, y_test, 'XGBoost', 'xgb'))

In [ ]:
pd.options.display.float_format = '{:_.2f}'.format
files = ["metrics/lr_metrics.pkl", "metrics/rf_metrics.pkl", "metrics/xgb_metrics.pkl"]

all_metrics = []

for file in files:
    with open(file, "rb") as f:
        all_metrics.append(pickle.load(f))

df = pd.DataFrame(all_metrics)
df

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10,6))
sns.barplot(x="model_name", y="MAE", data=df, color="skyblue", label="MAE")
sns.barplot(x="model_name", y="RMSE", data=df, color="salmon", label="RMSE", alpha=0.6)
plt.ylabel("Ошибка (рубли)")
plt.xticks(rotation=30)
plt.title("Сравнение MAE и RMSE моделей")
plt.legend()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(x="model_name", y="MAE", data=df, ax=axes[0])
axes[0].set_ylabel("Ошибка (рубли)")

sns.barplot(x="model_name", y="MAPE", data=df, ax=axes[1])
axes[1].set_ylabel("MAPE (%)")

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(x="model_name", y="R2", data=df, palette="viridis")
plt.ylabel("R^2")
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.title("Коэффициент детерминации R^2 для моделей")